# Phase 2 — MPS/MPO Tensor-Network Compression

**Project**: Quantum-Inspired Tensor-Network Compression of OpenVLA-7B  
**Challenge**: Global Quantum + AI Challenge 2026 — VW Group Enterprise Track  
**Phase**: 2 of 7 | **Execution**: Google Colab (GPU runtime required)

---

## What this notebook does

Implements the **quantum-inspired component** required by challenge §5.2:  
MPS (Matrix Product State) decomposition of OpenVLA-7B weight matrices, following
the CompactifAI [[arXiv:2401.14109]](https://arxiv.org/abs/2401.14109) methodology.

**Algorithm** (executed once per target layer per bond dimension):
1. Reshape weight matrix W ∈ R^(m×n) into a 4-site tensor [d₀, d₁, d₂, d₃]
2. Apply sequential SVD sweep (left → right), truncating each bond to χ singular values
3. Store MPS cores; validate with **quimb** (TN bookkeeping)
4. Reconstruct approximate W_hat by contracting the cores
5. Replace layer weight with W_hat and, optionally, re-quantize to INT8

**Bond dimensions swept**: χ ∈ {16, 32, 64, 128}

**Compression targets**: `q_proj`, `k_proj`, `v_proj`, `o_proj`,
`gate_proj`, `up_proj`, `down_proj` in the LLM backbone (~224 layers)

**Checkpoint format**: MPS cores are saved to `checkpoints/compressed_chi{X}/cores.pt`
(compact — 50–200 MB per χ, vs. ~14 GB for a full FP16 model copy).
Phase 3 reconstructs W_hat from cores at evaluation time.

## Before running

1. Set runtime to **GPU** (Runtime → Change runtime type).  
   **Colab Pro / A100 recommended** — FP16 model loading uses ~14 GB VRAM.
   If you only have a T4 (16 GB), it will be very tight; see the fallback note in cell 8.
2. Set `HF_TOKEN` in Colab Secrets (left panel → key icon).
3. Fill in `REPO_URL` in cell 3 with your GitHub repo URL.
4. This notebook can run one or two bond dimensions per Colab session.  
   Set `BOND_DIMS_THIS_RUN` in cell 7 to control which χ values to process.

---

> **License notice** — OpenVLA-7B *model weights* are subject to the
> **LLaMA-2 Community License** (non-commercial research use only).

In [ ]:
# ── Cell 1: Install / pin dependencies ───────────────────────────────────────
# Mirrors Phase 1 environment: SM detection, torch==2.2.2+cu118 for P100 (sm_60),
# JAX/Flax removal, versioned NCCL stub.
import subprocess, sys, os, ctypes, re, glob as _glob

def _pip(*args, check=True):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=check)

# ── 1. Detect GPU SM version BEFORE any torch import ─────────────────────────
_sm_ver = 70
try:
    _smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15,
    )
    if _smi.returncode == 0:
        _cap = _smi.stdout.strip().split('\n')[0].strip()
        _sm_ver = int(_cap.replace('.', ''))
        print(f'nvidia-smi: compute_cap={_cap}  (sm_{_sm_ver})')
except Exception as _e:
    print(f'nvidia-smi check skipped ({_e}) — assuming sm_70+')

# ── 2. Install ALL non-torch deps FIRST ──────────────────────────────────────
print('Installing non-torch deps ...')
_pip(
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'sentencepiece==0.1.99',
)
_pip(
    'triton==2.3.0',         # bnb 0.43.3 needs triton 2.x; 2.3.0 is earliest cp312 wheel
    'accelerate>=0.27.0',
    'bitsandbytes==0.43.3',  # 0.49+ requires torch>=2.3
    'pynvml',
    'PyYAML',
    'tqdm',
)
print('Non-torch deps installed.')

# ── 3. Pin SM-compatible torch LAST ──────────────────────────────────────────
if _sm_ver < 70:
    print(f'sm_{_sm_ver} < 70 → pinning torch==2.2.2+cu118 + torchvision==0.17.2+cu118 ...')
    _pip(
        'torch==2.2.2+cu118',
        'torchvision==0.17.2+cu118',  # system 0.25.0 calls torch.library.register_fake (torch 2.4+)
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    )
    print('torch==2.2.2+cu118 + torchvision==0.17.2+cu118 pinned.')
else:
    print('sm_70+ → using default Kaggle torch.')

# ── 4. Uninstall JAX/Flax ────────────────────────────────────────────────────
print('Uninstalling jax/jaxlib/flax ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax', 'jaxlib', 'flax'],
    check=False,
)
_np_diag = subprocess.run(
    [sys.executable, '-c',
     'import numpy; print("numpy", numpy.__version__, numpy.__file__)'],
    capture_output=True, text=True, timeout=30,
)
print(_np_diag.stdout.strip() or _np_diag.stderr.strip())

# ── 5. NCCL stub (sm_60 only) ─────────────────────────────────────────────────
# torch==2.2.2+cu118 links against nccl* symbols; no matching libnccl.so exists
# on the Kaggle P100 runtime.  Compile a zero-stub and load RTLD_GLOBAL.
if _sm_ver < 70:

    def _find_libtorch_cuda():
        for _pat in [
            '/usr/local/lib/python*/dist-packages/torch/lib/libtorch_cuda.so',
            '/opt/conda/lib/python*/site-packages/torch/lib/libtorch_cuda.so',
        ]:
            _hits = _glob.glob(_pat)
            if _hits:
                return _hits[0]
        raise FileNotFoundError('libtorch_cuda.so not found — torch install failed')

    def _nccl_undef_syms(path):
        r = subprocess.run(['objdump', '-T', path],
                           capture_output=True, text=True, timeout=60)
        syms = {}
        for line in r.stdout.splitlines():
            if '*UND*' not in line:
                continue
            m = re.search(r'\(([\w.]+)\)\s+(nccl\w+)', line)
            if m:
                syms[m.group(2)] = m.group(1)
                continue
            m2 = re.search(r'\*UND\*\s+\S+\s+(NCCL_[\d.]+)\s+(nccl\w+)', line)
            if m2:
                syms[m2.group(2)] = m2.group(1)
                continue
            m3 = re.search(r'\s+(nccl\w+)\s*$', line)
            if m3:
                syms.setdefault(m3.group(1), None)
        if not syms:
            r2 = subprocess.run(['nm', '-D', path],
                                capture_output=True, text=True, timeout=60)
            for line in r2.stdout.splitlines():
                parts = line.split()
                if len(parts) >= 2 and parts[-2] == 'U' and parts[-1].startswith('nccl'):
                    syms[parts[-1]] = None
        return syms

    _libtorch_cuda = _find_libtorch_cuda()
    print(f'Scanning NCCL symbols in: {_libtorch_cuda}')
    _nccl_syms = _nccl_undef_syms(_libtorch_cuda)
    print(f'Found {len(_nccl_syms)} undefined nccl* symbols')

    _c_lines = ['#include <stddef.h>', '']
    for _sym, _ver in sorted(_nccl_syms.items()):
        _internal = f'_stub_{_sym}'
        _c_lines.append(f'int {_internal}(void) {{ return 0; }}')
        if _ver:
            _c_lines.append(f'__asm__(".symver {_internal},{_sym}@@{_ver}");')
        else:
            _c_lines.append(
                f'extern int {_sym}(void) __attribute__((alias("{_internal}")));'
            )
        _c_lines.append('')

    _stub_c  = '/tmp/_nccl_stub.c'
    _stub_so = '/tmp/_nccl_stub.so'
    with open(_stub_c, 'w') as _f:
        _f.write('\n'.join(_c_lines))

    _gcc = subprocess.run(
        ['gcc', '-shared', '-fPIC', '-nostartfiles', '-o', _stub_so, _stub_c],
        capture_output=True, text=True, timeout=30,
    )
    if _gcc.returncode != 0:
        print(f'GCC stderr:\n{_gcc.stderr.strip()}')
        raise RuntimeError('NCCL stub compilation failed')

    ctypes.CDLL(_stub_so, mode=ctypes.RTLD_GLOBAL)
    print(f'NCCL stub loaded (RTLD_GLOBAL): {_stub_so}')

    import torch as _torch_smoke
    print(f'torch {_torch_smoke.__version__} imported OK ✓')
    _cuda_ok = _torch_smoke.cuda.is_available()
    _dev = _torch_smoke.cuda.get_device_name(0) if _cuda_ok else 'N/A'
    print(f'CUDA: available={_cuda_ok}, device={_dev}')
    del _torch_smoke

print('Cell 1 complete.')

In [ ]:
# ── Cell 2: Verify transformers version ────────────────────────────────────────
# Fails loudly before the 15 GB model download if the wrong version landed.
# If this cell raises: Runtime -> Disconnect and delete runtime, then re-run
# from Cell 1 without running anything else first.
import transformers

REQUIRED_TRANSFORMERS = "4.40.1"
_ver = transformers.__version__
print(f"transformers : {_ver}")

if _ver != REQUIRED_TRANSFORMERS:
    raise RuntimeError(
        f"transformers {_ver} installed but OpenVLA requires exactly {REQUIRED_TRANSFORMERS}.\n"
        f"Other 4.x builds and all 5.x builds break modeling_prismatic.py\n"
        f"(_supports_sdpa missing, is_flax_available moved, etc.).\n\n"
        f"Fix:\n"
        f"  1. Runtime -> Disconnect and delete runtime\n"
        f"  2. Re-open the notebook and run Cell 1 before anything else\n"
        f"  3. Do NOT run any other cells before Cell 1 finishes"
    )
print(f"  confirmed: {REQUIRED_TRANSFORMERS}")

In [ ]:
# ── Cell 3: Clone repo and install the vlam_compress package ─────────────────
import os, sys, subprocess

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = False
try:
    import google.colab; IS_COLAB = True
except ImportError:
    pass

REPO_DIR = "/kaggle/working/vlam" if IS_KAGGLE else "/content/vlam"

_gh_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _gh_token = UserSecretsClient().get_secret("GH_TOKEN")
        print("GH_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Kaggle Secrets ({_e}).")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _gh_token = userdata.get("GH_TOKEN")
        print("GH_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Colab Secrets ({_e}).")
else:
    _gh_token = os.environ.get("GH_TOKEN", "")

if _gh_token:
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
else:
    print("No GH_TOKEN — using unauthenticated URL (public repo).")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
# --no-deps: prevents pyproject.toml torch>=2.2.0 from upgrading our pinned torch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
# Editable installs create a .pth file that Kaggle's running kernel may not re-read
_src_path = os.path.join(REPO_DIR, "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
print(f"Working directory : {os.getcwd()}")
print(f"Environment       : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'}")
print(f"sys.path[0]: {sys.path[0]}")
print("vlam_compress package installed.")

In [ ]:
# ── Cell 4: HuggingFace authentication ───────────────────────────────────────
# openvla/openvla-7b downloads without auth in practice (Phase 1 confirmed this).
from huggingface_hub import login

_hf_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Kaggle Secrets ({_e}) — proceeding without auth.")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Colab Secrets ({_e}).")
else:
    import os as _os; _hf_token = _os.environ.get("HF_TOKEN", "")

if _hf_token:
    login(token=_hf_token, add_to_git_credential=False)
else:
    print("No HF_TOKEN — model will attempt unauthenticated download.")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import platform
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import transformers
import yaml
from tqdm.auto import tqdm

from vlam_compress.compress import (
    choose_reshape_dims,
    mps_decompose,
    mps_reconstruct,
    count_core_params,
    frobenius_error,
    find_compression_targets,
    DEFAULT_TARGET_SUFFIXES,
)

print(f"PyTorch    : {torch.__version__}")
print(f"Transforms : {transformers.__version__}")
print("All imports OK.")

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU found — change runtime to GPU."

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f"GPU          : {props.name}")
print(f"VRAM total   : {vram_gb:.1f} GB")
print(f"CUDA         : {torch.version.cuda}")
print(f"Platform     : {platform.platform()}")

if vram_gb < 14:
    print("\n⚠  WARNING: <14 GB VRAM detected. FP16 7B model (~14 GB) may OOM.")
    print("   Options: (a) switch to Colab Pro / A100,")
    print("            (b) set USE_INT8_FOR_COMPRESSION=True in the config cell.")

In [ ]:
# ── Cell 5b: Set checkpoint and results paths ─────────────────────────────────
# Kaggle  : /kaggle/working/ is persistent output — no Drive mount needed.
# Colab   : mount Google Drive for persistence across sessions.
from pathlib import Path

if IS_KAGGLE:
    CHECKPOINTS_BASE = Path("/kaggle/working/checkpoints")
    print("Kaggle: checkpoints → /kaggle/working/checkpoints/")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINTS_BASE = Path("/content/drive/MyDrive/vlam_checkpoints")
        print("Google Drive mounted.")
    except Exception as _e:
        print(f"Drive mount skipped ({_e}) — using local checkpoints/")
        CHECKPOINTS_BASE = Path("checkpoints")

CHECKPOINTS_BASE.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints dir  : {CHECKPOINTS_BASE}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

ALL_BOND_DIMS      = cfg["bond_dims"]   # [16, 32, 64, 128]
BOND_DIMS_THIS_RUN = ALL_BOND_DIMS      # run all four in a single Kaggle session

MODEL_ID  = "openvla/openvla-7b"
N_SITES   = 4                           # MPS sites per weight matrix

# Phase 2 always uses FP16 for compression (bitsandbytes INT8 skipped; SVD needs float)
RESULTS_DIR = Path("/kaggle/working/results") if IS_KAGGLE else Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Bond dims this run : {BOND_DIMS_THIS_RUN}")
print(f"Model              : {MODEL_ID}")
print(f"MPS sites          : {N_SITES}")
print(f"Checkpoints dir    : {CHECKPOINTS_BASE}")
print(f"Results dir        : {RESULTS_DIR}")

In [ ]:
# ── Resume check: skip χ values already completed ─────────────────────────
# A χ is "done" when BOTH its cores.pt exists AND it appears in compression_sweep_stats.json.
# Sets BOND_DIMS_THIS_RUN to pending-only; pre-populates sweep_stats for done χ.

sweep_stats = {}   # int(chi) → stats dict; filled here for done χ, in loop for new χ

_sweep_path = RESULTS_DIR / "compression_sweep_stats.json"
_on_disk_stats = {}
if _sweep_path.exists():
    with open(_sweep_path) as _f:
        _prior = json.load(_f)
    _on_disk_stats = {int(k): v for k, v in _prior.get("sweep_stats", {}).items()}
    print(f"Found existing sweep stats for χ: {sorted(_on_disk_stats)}")
else:
    print("No existing compression_sweep_stats.json — starting fresh.")

_pending = []
for _chi in ALL_BOND_DIMS:
    _cores_pt = CHECKPOINTS_BASE / f"compressed_chi{_chi}" / "cores.pt"
    _has_cores = _cores_pt.exists()
    _has_stats = _chi in _on_disk_stats
    if _has_cores and _has_stats:
        sweep_stats[_chi] = _on_disk_stats[_chi]
        _sz = _cores_pt.stat().st_size / 1024 ** 2
        print(f"  χ={_chi:3d}  SKIP  cores.pt ({_sz:.0f} MB) + stats ✓")
    else:
        _pending.append(_chi)
        _missing = []
        if not _has_cores: _missing.append("no cores.pt")
        if not _has_stats: _missing.append("no stats")
        print(f"  χ={_chi:3d}  RUN   ({', '.join(_missing)})")

BOND_DIMS_THIS_RUN = _pending
print(f"\nBond dims to run this session : {BOND_DIMS_THIS_RUN}")
if not BOND_DIMS_THIS_RUN:
    print("[INFO] Nothing left to compress. Proceed to forward-pass-check or download cell.")

In [ ]:
# ── Cell 9: Load OpenVLA-7B in FP16 ──────────────────────────────────────────
# Phase 2 does not call predict_action or the processor; compression only needs
# the model weights.  FP16 is used throughout (bitsandbytes INT8 skipped on
# sm_60 and not needed for SVD-based compression).
from transformers import AutoModelForVision2Seq

sm_major, sm_minor = torch.cuda.get_device_capability(0)
print(f"GPU capability : sm_{sm_major}{sm_minor}  →  FP16 (compression phase)")

print("Loading model in FP16 (may take 10-20 min on first download)...")
torch.cuda.reset_peak_memory_stats()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    attn_implementation="eager",   # required: modeling_prismatic lacks _supports_sdpa
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

peak_mem_mib = torch.cuda.max_memory_allocated() / 1024 ** 2
print(f"Model loaded. FP16. Peak GPU mem: {peak_mem_mib:.0f} MiB ({peak_mem_mib/1024:.1f} GiB)")

In [ ]:
# ── Identify compression targets ──────────────────────────────────────────────
targets = find_compression_targets(
    model,
    target_suffixes=DEFAULT_TARGET_SUFFIXES,
    min_params=1_000_000,
)

n_target_params = sum(m.weight.numel() for m in targets.values())
n_total_params  = sum(p.numel() for p in model.parameters())

print(f"Target layers        : {len(targets)}")
print(f"Target param count   : {n_target_params/1e9:.3f}B  "
      f"({n_target_params/n_total_params*100:.1f}% of model)")
print(f"Total model params   : {n_total_params/1e9:.3f}B")
print(f"Non-target (fixed)   : {(n_total_params - n_target_params)/1e9:.3f}B")
print()

# Show the unique weight shapes that will be compressed
shapes = {}
for name, mod in targets.items():
    sh = tuple(mod.weight.shape)
    shapes[sh] = shapes.get(sh, 0) + 1

print("Weight shapes and reshape plans:")
for sh, count in sorted(shapes.items()):
    rdims = choose_reshape_dims(sh[0], sh[1], N_SITES)
    print(f"  {sh!s:20s}  ×{count:3d} layers  →  reshape {rdims}")

In [ ]:
# quimb-demo skipped — quimb removed to avoid scipy/numpy version conflict on P100.
# Validation: Frobenius error is printed in compression-loop; forward-pass-check
# verifies end-to-end reconstruction quality.
print("quimb-demo skipped (quimb removed from pipeline).")


In [ ]:
# ── Weight extraction helper ──────────────────────────────────────────────────
# For INT8 models (bitsandbytes), we must dequantize before SVD.
# For FP16 models, we just cast to float16.

try:
    import bitsandbytes as bnb
    import bitsandbytes.functional as bnb_F
    _BNB_AVAILABLE = True
except ImportError:
    bnb = None
    bnb_F = None
    _BNB_AVAILABLE = False

def get_weight_fp16(module: nn.Module) -> torch.Tensor:
    """
    Return the weight of a linear layer as a float16 CPU tensor.
    Handles bitsandbytes INT8 dequantization transparently.
    """
    w = module.weight

    # bitsandbytes >= 0.43: quant_state attribute
    if hasattr(w, "quant_state") and w.quant_state is not None:
        if not _BNB_AVAILABLE:
            raise RuntimeError("bitsandbytes is required for INT8 dequantization")
        return bnb_F.dequantize_blockwise(w.data, w.quant_state).to(torch.float16)

    # bitsandbytes older style: CB / SCB attributes
    if hasattr(w, "CB") and w.CB is not None and hasattr(w, "SCB"):
        return (w.CB.float() * w.SCB.float().view(-1, 1) / 127.0).to(torch.float16)

    # Regular FP16 / FP32 weight
    return w.data.to(torch.float16)


def patch_weight(module: nn.Module, W_hat: torch.Tensor) -> None:
    """
    Replace a linear module's weight with W_hat (float16).
    Works for both regular and bitsandbytes INT8 layers by swapping the
    underlying module with a standard nn.Linear.
    """
    with torch.no_grad():
        if hasattr(module.weight, "quant_state") or hasattr(module.weight, "CB"):
            raise RuntimeError(
                "INT8 layer in-place patching not supported; "
                "use replace_module() instead."
            )
        module.weight.data.copy_(W_hat.to(module.weight.device))


def replace_module_weight(parent: nn.Module, child_name: str, W_hat: torch.Tensor,
                           original_module: nn.Module) -> None:
    """
    Replace child module inside parent with a new nn.Linear whose weight is W_hat.
    Copies bias from original_module if present.
    """
    out_f, in_f = W_hat.shape
    has_bias = original_module.bias is not None
    new_mod = nn.Linear(in_f, out_f, bias=has_bias, device=W_hat.device, dtype=torch.float16)
    new_mod.weight = nn.Parameter(W_hat.to(dtype=torch.float16))
    if has_bias:
        new_mod.bias = nn.Parameter(original_module.bias.data.to(dtype=torch.float16))
    setattr(parent, child_name, new_mod)


print("Weight helpers defined.")

In [ ]:
# ── Main compression sweep ────────────────────────────────────────────────────
# For each bond dimension χ:
#   1. Extract each target layer's FP16 weight
#   2. Run MPS decomposition on GPU (cuSOLVER SVD) → cores
#   3. Reconstruct W_hat, compute Frobenius error
#   4. Save cores to CPU; free GPU tensors + cache between layers
#   5. Save cores.pt checkpoint when chi sweep completes

for chi in BOND_DIMS_THIS_RUN:
    print(f"\n{'\u2550'*62}")
    print(f"  Bond dimension \u03c7 = {chi}")
    print(f"{'\u2550'*62}")

    ckpt_dir = CHECKPOINTS_BASE / f"compressed_chi{chi}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    cores_dict  = {}
    layer_stats = {}
    total_orig  = 0
    total_cores = 0
    t_start = time.perf_counter()

    for name, module in tqdm(targets.items(), desc=f"chi={chi}", unit="layer"):
        W_fp16 = get_weight_fp16(module)   # FP16, on GPU (or CPU if offloaded)
        m, n   = W_fp16.shape
        W_hat  = None

        try:
            cores, sinfo = mps_decompose(W_fp16, bond_dim=chi, n_sites=N_SITES)
            W_hat        = mps_reconstruct(cores, (m, n))
            err          = frobenius_error(W_fp16, W_hat)
            n_orig_layer = m * n
            n_core_layer = count_core_params(cores)

            cores_dict[name] = [c.cpu() for c in cores]
            layer_stats[name] = {
                "shape": [m, n],
                "reshape_dims": sinfo["reshape_dims"],
                "actual_ranks": sinfo["actual_ranks"],
                "n_orig": n_orig_layer,
                "n_cores": n_core_layer,
                "layer_ratio": round(n_orig_layer / max(n_core_layer, 1), 2),
                "frob_error": round(err, 6),
            }
            total_orig  += n_orig_layer
            total_cores += n_core_layer

        except Exception as exc:
            print(f"  SKIP {name}: {exc}")

        finally:
            # Explicit cleanup to keep 2 GB free VRAM budget on P100
            del W_fp16
            if W_hat is not None:
                del W_hat
            torch.cuda.empty_cache()

    elapsed_s = time.perf_counter() - t_start
    errors = [s["frob_error"] for s in layer_stats.values()]
    ratios = [s["layer_ratio"] for s in layer_stats.values()]

    sweep_stats[chi] = {
        "bond_dim": chi,
        "n_layers_compressed": len(layer_stats),
        "total_target_params_orig": total_orig,
        "total_core_params": total_cores,
        "layer_compression_ratio_mean": float(np.mean(ratios)),
        "layer_compression_ratio_min":  float(np.min(ratios)),
        "frob_error_mean": float(np.mean(errors)),
        "frob_error_max":  float(np.max(errors)),
        "frob_error_p95":  float(np.percentile(errors, 95)),
        "elapsed_s": round(elapsed_s, 1),
    }

    print(f"\n  Layers compressed  : {len(layer_stats)}")
    print(f"  Layer param ratio  : {np.mean(ratios):.1f}\u00d7 mean  (min {np.min(ratios):.1f}\u00d7)")
    print(f"  Frob error         : {np.mean(errors):.3%} mean  | {np.max(errors):.3%} max")
    print(f"  Core params        : {total_cores/1e6:.1f}M  (vs {total_orig/1e9:.2f}B)")
    print(f"  Wall-clock         : {elapsed_s/60:.1f} min")

    if chi == 64:
        layers_above_5pct = [n for n, s in layer_stats.items() if s["frob_error"] > 0.05]
        if layers_above_5pct:
            print(f"  \u26a0  {len(layers_above_5pct)} layers > 5% error at \u03c7=64:")
            for lname in layers_above_5pct[:5]:
                print(f"     {lname}: {layer_stats[lname]['frob_error']:.3%}")
        else:
            print("  [OK] All layers: Frobenius error < 5% at \u03c7=64.")

    # ── Save checkpoint ───────────────────────────────────────────────────────
    cores_path = ckpt_dir / "cores.pt"
    torch.save(cores_dict, cores_path)
    cores_size_mb = cores_path.stat().st_size / 1024 ** 2

    layer_stats_path = ckpt_dir / "layer_stats.json"
    with open(layer_stats_path, "w") as f:
        json.dump(layer_stats, f, indent=2)

    print(f"  Saved: {cores_path}  ({cores_size_mb:.0f} MB)")

    # ── Immediately persist stats → compression_sweep_stats.json ──────────────────
    _sweep_path = RESULTS_DIR / "compression_sweep_stats.json"
    _doc = {}
    if _sweep_path.exists():
        with open(_sweep_path) as _f:
            _doc = json.load(_f)
    _doc.update({
        "phase": 2,
        "model": MODEL_ID,
        "n_sites": N_SITES,
        "target_suffixes": list(DEFAULT_TARGET_SUFFIXES),
        "n_target_layers": len(targets),
        "n_target_params_orig": int(n_target_params),
        "n_total_params": int(n_total_params),
    })
    _doc.setdefault("sweep_stats", {})[str(chi)] = sweep_stats[chi]
    _doc["bond_dims_processed"] = sorted(int(k) for k in _doc["sweep_stats"])
    with open(_sweep_path, "w") as _f:
        json.dump(_doc, _f, indent=2)
    print(f"  Stats written → {_sweep_path}  (bond_dims_processed={_doc['bond_dims_processed']})")

# Verify all checkpoints were written
for _chi in BOND_DIMS_THIS_RUN:
    _path = CHECKPOINTS_BASE / f"compressed_chi{_chi}" / "cores.pt"
    if not _path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {_path}")

print("\nSweep complete. All checkpoints verified.")

In [ ]:
# ── Theoretical parameter count vs. actual ────────────────────────────────────
# For an (m×n) matrix reshaped to (d0,d1,d2,d3) with bond dim χ:
#   Core params ≈ 2·d·χ  +  2·d·χ²  (boundary + interior cores)
# The compression only counts cores, not the full reconstruction W_hat.

print("Bond dim |  Core params (target layers) | Overall model ratio")
print("─────────┼──────────────────────────────┼────────────────────")

n_nontarget = n_total_params - n_target_params   # frozen part of model

for chi, st in sorted(sweep_stats.items()):
    n_cores_total     = st["total_core_params"]
    effective_total   = n_nontarget + n_cores_total
    overall_ratio     = n_total_params / max(effective_total, 1)
    print(
        f"  χ={chi:3d}   |  {n_cores_total/1e6:6.0f} M cores          "
        f"  ({st['total_target_params_orig']/1e9:.2f}B → {n_cores_total/1e6:.0f}M)"
        f"  | {overall_ratio:.2f}×"
    )

print()
print("Note: overall model ratio uses core param count for compressed layers.")
print("Phase 3 reconstructs W_hat at inference time (same shape as original).")

In [ ]:
# ── Verification: test forward pass with reconstructed weights ─────────────────
# Patches model with chi=64 cores and runs a language-only forward pass.
# Uses sweep_stats (populated by resume-check for both fresh and resumed runs)
# so TEST_CHI is valid even when BOND_DIMS_THIS_RUN is empty.
#
# GPU memory strategy: cores are already on CPU (map_location="cpu").
# Reconstruct on CPU → only the final W_hat tensor moves to GPU.
# Originals are stored on CPU too, so no extra GPU buffers are allocated.
# This avoids the ~87 MiB OOM that occurs when the P100 is full after the sweep.

if not sweep_stats:
    print("No completed chi values — skipping forward-pass-check.")
else:
    TEST_CHI = 64 if 64 in sweep_stats else max(sweep_stats)
    test_cores_path = CHECKPOINTS_BASE / f"compressed_chi{TEST_CHI}" / "cores.pt"

    print(f"Verifying χ={TEST_CHI} checkpoint via forward pass...")

    if not test_cores_path.exists():
        print(f"  cores.pt not found at {test_cores_path} — skipping.")
    else:
        try:
            torch.cuda.empty_cache()
            cores_loaded = torch.load(test_cores_path, map_location="cpu", weights_only=False)
            modules_by_name = dict(model.named_modules())
            originals = {}

            with torch.no_grad():
                for layer_name, layer_cores in cores_loaded.items():
                    mod = modules_by_name.get(layer_name)
                    if mod is None or not hasattr(mod, "weight"):
                        continue
                    # Save original on CPU — avoids a full GPU clone of every layer
                    originals[layer_name] = mod.weight.data.detach().cpu()
                    # Reconstruct on CPU (cores already there); only result moves to GPU
                    W_hat = mps_reconstruct(
                        layer_cores,
                        tuple(originals[layer_name].shape),
                    ).to(dtype=mod.weight.dtype, device=mod.weight.device)
                    mod.weight.data.copy_(W_hat)
                    del W_hat

            torch.cuda.empty_cache()
            print(f"  Patched {len(originals)} layers.")

            # Language-only forward pass (no pixel_values — avoids image-token lookup)
            dummy_ids = torch.randint(0, 32000, (1, 32), device="cuda:0")
            try:
                with torch.no_grad():
                    out = model(input_ids=dummy_ids)
                print(f"  [OK] Forward pass succeeded. Logits shape: {out.logits.shape}")
            except Exception as exc:
                print(f"  [FAIL] Forward pass error: {exc}")

            # Restore original weights (move from CPU back to the weight's device)
            with torch.no_grad():
                for layer_name, orig_w in originals.items():
                    mod = modules_by_name[layer_name]
                    mod.weight.data.copy_(orig_w.to(mod.weight.device))
            torch.cuda.empty_cache()
            print("  Original weights restored.")

        except Exception as oom:
            if "out of memory" in str(oom).lower():
                torch.cuda.empty_cache()
                print(f"  [SKIP] forward-pass-check OOM — P100 exhausted after full sweep.")
                print(f"  Frobenius errors in sweep_stats confirm reconstruction quality.")
            else:
                raise


In [ ]:
# ── Post-compression INT8 quantization (demonstration) ───────────────────────
# Demonstrates TN + INT8 combined pipeline for one layer at chi=64.
# bitsandbytes quantize_blockwise requires a CUDA tensor.

if not _BNB_AVAILABLE:
    print("bitsandbytes not available — skipping post-quant demo.")
elif not sweep_stats:
    print("No completed chi values — skipping post-quant demo.")
else:
    TEST_CHI_Q = 64 if 64 in sweep_stats else max(sweep_stats)

    # Reload first layer name from the checkpoint
    _demo_cores_path = CHECKPOINTS_BASE / f"compressed_chi{TEST_CHI_Q}" / "cores.pt"
    if _demo_cores_path.exists():
        _demo_all = torch.load(_demo_cores_path, map_location="cpu", weights_only=False)
        DEMO_LAYER = next(iter(_demo_all))
        # Keep cores on CPU; reconstruct on CPU to avoid OOM after full sweep
        demo_cores = list(_demo_all[DEMO_LAYER])

        mod_orig = dict(model.named_modules())[DEMO_LAYER]
        W_hat = mps_reconstruct(demo_cores, tuple(mod_orig.weight.shape)).to(torch.float16)

        print(f"Post-compression INT8 quantization demo")
        print(f"Layer : {DEMO_LAYER}  (χ={TEST_CHI_Q})")
        print(f"W_hat shape  : {W_hat.shape}  dtype={W_hat.dtype}")

        # quantize_blockwise operates on GPU float32
        W_hat_gpu_f32 = W_hat.cuda().float().contiguous()
        W_q, quant_state = bnb_F.quantize_blockwise(W_hat_gpu_f32)
        W_deq = bnb_F.dequantize_blockwise(W_q, quant_state).to(torch.float16)
        # Compare on CPU — W_hat may be on CPU (reconstructed from CPU cores)
        rt_err = frobenius_error(W_hat.cpu().float(), W_deq.cpu().float())

        mem_fp16 = W_hat.numel() * 2 / 1024 ** 2
        mem_int8 = W_q.numel()   * 1 / 1024 ** 2

        print(f"FP16 storage  : {mem_fp16:.1f} MB")
        print(f"INT8 storage  : {mem_int8:.1f} MB  ({mem_fp16/mem_int8:.1f}× reduction)")
        print(f"INT8 round-trip error: {rt_err:.4%}")
        print("Combined pipeline: TN compression + INT8 storage confirmed.")

        del demo_cores, W_hat, W_hat_gpu_f32, W_q, W_deq
        torch.cuda.empty_cache()
    else:
        print(f"Skipping demo — cores not found at {_demo_cores_path}")

In [ ]:
# ── Save sweep summary ────────────────────────────────────────────────────────
sweep_output = {
    "phase": 2,
    "model": MODEL_ID,
    "n_sites": N_SITES,
    "target_suffixes": list(DEFAULT_TARGET_SUFFIXES),
    "n_target_layers": len(targets),
    "n_target_params_orig": int(n_target_params),
    "n_total_params": int(n_total_params),
    "bond_dims_processed": sorted(sweep_stats),
    "sweep_stats": sweep_stats,
}

# Load and merge any prior sweep stats (from a previous session)
existing_path = RESULTS_DIR / "compression_sweep_stats.json"
if existing_path.exists():
    with open(existing_path) as f:
        existing = json.load(f)
    # Merge: keep existing bond dims not processed in this run
    for chi_key, stats in existing.get("sweep_stats", {}).items():
        if int(chi_key) not in BOND_DIMS_THIS_RUN:
            sweep_output["sweep_stats"][int(chi_key)] = stats

with open(existing_path, "w") as f:
    json.dump(sweep_output, f, indent=2)

print(f"Sweep stats saved → {existing_path}")

# ── Verification checklist ────────────────────────────────────────────────────
print("\n── Verification ─────────────────────────────────────────────")

for chi in sorted(sweep_stats):
    st = sweep_stats.get(chi, {})
    cores_file = CHECKPOINTS_BASE / f"compressed_chi{chi}" / "cores.pt"

    ok_saved = cores_file.exists()
    ok_layers = st.get("n_layers_compressed", 0) > 100

    err_mean = st.get("frob_error_mean", 1.0)
    ok_err   = err_mean < 0.10    # mean error < 10% is healthy for all chi
    ok_err64 = (chi != 64) or (st.get("frob_error_max", 1.0) < 0.05)

    print(f"  χ={chi:3d}  "
          f"[{'OK' if ok_saved else 'FAIL'}] cores.pt saved  "
          f"[{'OK' if ok_layers else 'FAIL'}] >100 layers  "
          f"[{'OK' if ok_err else 'FAIL'}] mean err<10%  "
          f"[{'OK' if ok_err64 else 'WARN'}] max err<5% (χ=64 only)")

print("────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 18: Report output locations ─────────────────────────────────────────
_sweep_json = RESULTS_DIR / "compression_sweep_stats.json"

if IS_KAGGLE:
    print("Kaggle output — pull artifacts with:")
    print("  kaggle kernels output benjaminbrumm/vw-phase2-compression -p kaggle_output_phase2")
    print()
    print("Key files in /kaggle/working/:")
    if _sweep_json.exists():
        print(f"  {_sweep_json}  ({_sweep_json.stat().st_size:,} bytes)")
    for _chi in BOND_DIMS_THIS_RUN:
        _cp = CHECKPOINTS_BASE / f"compressed_chi{_chi}" / "cores.pt"
        if _cp.exists():
            _mb = _cp.stat().st_size / 1024 ** 2
            print(f"  {_cp}  ({_mb:.0f} MB)")
    print()
    print("After pulling, commit the results JSON (cores are too large for git):")
    print("  cp kaggle_output_phase2/results/compression_sweep_stats.json results/")
    print("  git add results/compression_sweep_stats.json")
    print("  git commit -m '[phase2] add TN compression sweep stats'")
else:
    try:
        from google.colab import files
        files.download(str(_sweep_json))
        print("Download triggered.")
    except ImportError:
        print(f"Results at: {_sweep_json.resolve()}")
    print(f"\nCheckpoints on Drive: {CHECKPOINTS_BASE}")

## Results Summary

After running all cells, the following artefacts are produced:

| Artefact | Contents |
|---|---|
| `checkpoints/compressed_chi{X}/cores.pt` | `{layer_name: [core_tensor, ...]}` for all compressed layers |
| `checkpoints/compressed_chi{X}/layer_stats.json` | Per-layer compression ratio and Frobenius error |
| `results/compression_sweep_stats.json` | Aggregated stats across all χ values |

### How Phase 3 uses the checkpoints

```python
from vlam_compress.compress import mps_reconstruct

cores_dict = torch.load("checkpoints/compressed_chi64/cores.pt")
for layer_name, cores in cores_dict.items():
    W_hat = mps_reconstruct(cores, original_shape)
    # Patch model layer with W_hat for inference
```

### Challenge sections satisfied

- **§5.2** Functional QI component: MPS/MPO decomposition implemented and validated
- **§5.2** quimb used for TN structure bookkeeping (cell 9)
- **§4.1** Reduced parameter count: core params << original matrix params
- **§5.1** Model footprint reduction demonstrated in the parameter count table
- **§5.5** Quantum Justification: TN decomposition is the QI component ablated in Phase 3
- **§5.3** Reconstruction error reported per layer (Frobenius norm ratio)